In [2]:
import sys
sys.path.append("..")

import json
from pathlib import Path

from chat2edit.context.attachments import Attachment
from core.chat2edit.models import Image
from core.chat2edit.strategies.mic2e_context_strategy import Mic2eContextStrategy

# Load test image from JSON (same as in test_functions.ipynb)
json_path = Path("../resources/images/test2.json")
with open(json_path, "r") as f:
    data = json.load(f)

image = Image.model_validate(data)
print(f"Loaded image: {image.width}x{image.height}")

# Build a sample context similar to what Mic2eContextStrategy will see in real runs
context = {
    "image_1": Attachment(image),  # main image attachment
}

print("Original context keys:", list(context.keys()))

strategy = Mic2eContextStrategy()
filtered = strategy.filter_context(context)

print("Filtered context keys:", list(filtered.keys()))
print("Filtered values types:", {k: type(v) for k, v in filtered.items()})



Loaded image: 270.0x148.0
Original context keys: ['image_1']
Filtered context keys: []
Filtered values types: {}


In [4]:
from typing import Any, Dict, List, Union

from pydantic import TypeAdapter

from chat2edit.context.attachments import Attachment
from core.chat2edit.models import Box, Image, Object, Point, Scribble, Text

CONTEXT_VALUE_BASE_TYPE = Union[
    Image,
    Object,
    Box,
    Point,
    Text,
    Scribble,
    int,
    str,
    float,
    bool,
]
CONTEXT_ITEM_TYPE = Union[CONTEXT_VALUE_BASE_TYPE, List[CONTEXT_VALUE_BASE_TYPE]]


def debug_filter_context(context: Dict[str, Any]) -> Dict[str, Any]:
    """Standalone copy of Mic2eContextStrategy.filter_context for debugging.

    You can modify this implementation in the notebook and re-run to see
    how different rules affect what survives in the context.
    """

    filtered_context: Dict[str, Any] = {}
    item_adapter = TypeAdapter(CONTEXT_ITEM_TYPE)

    for key, value in context.items():
        base_value = value.__obj__ if isinstance(value, Attachment) else value

        try:
            item_adapter.validate_python(base_value)
            filtered_context[key] = base_value
        except Exception as e:
            print(f"Dropping key={key!r} type={type(value)} because: {e}")
            continue

    return filtered_context


dbg_filtered = debug_filter_context(context)
print("debug_filter_context keys:", list(dbg_filtered.keys()))
print("debug_filter_context types:", {k: type(v) for k, v in dbg_filtered.items()})



debug_filter_context keys: ['image_1']
debug_filter_context types: {'image_1': <class 'core.chat2edit.models.image.Image'>}
